In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys
import django

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [ ]:
from neural_data.models import *
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.utils import (
    show_single_channel_red_green_black as S,
    to_show_list as tsl,
    show_72,
    show_72_list,
    scatter_plot_1d,
    mk_rect_on_ax,
    get_receptive,
    otsu_threshold,
    explain_variance_with_pca,
)
from tqdm import tqdm
from pt_to_api.contribs.v1 import (
    show_input_patch_and_kernel_placement_for_poi_using_raw_params as SIP,
)
import seaborn as sns
import numpy as np
from sklearn.decomposition import MiniBatchDictionaryLearning, DictionaryLearning
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import (
    pairwise_distances,
    cosine_similarity,
    cosine_distances,
)
from scipy.optimize import linear_sum_assignment
import pandas as pd

from neural_data.utils import (
    get_full_conv_kernel_at_coordinate,
    get_saliency_map_ids_and_patches,
    get_full_activations_of_layer,
)
from collections import defaultdict
import itertools

In [ ]:
KERNEL_COORDINATE = "layers.2.out_12"
INPUT_LAYER_NAME = "layers.1"
MODE = "light"
R = 2
C = 3

# Data gather

In [ ]:
kernel = get_full_conv_kernel_at_coordinate(KERNEL_COORDINATE)
vals = []
poi_by_vals = defaultdict(list)
for sm in tqdm(SaliencyMap.objects.filter(coordinate=KERNEL_COORDINATE)):
    poi_by_vals[(R,C)].append(sm.data[R][C])
    # for i in range(len(sm.data)):
    #     for j in range(len(sm.data[i])):
    #         poi_by_vals[(i, j)].append(sm.data[i][j])

# scatter_plot_1d(vals)

In [ ]:
# oopsie doopsie, there is no distribution lol
# is this even maneagable?
# im at the same problem where i need to cluster manually
vals = list(itertools.chain.from_iterable(poi_by_vals.values()))

print("all")
pvals = [v for v in vals if v > 0]
nvals = [v for v in vals if v < 0]
pos_thresh = otsu_threshold(pvals)
neg_thresh = otsu_threshold(nvals)
pos_thresh, neg_thresh

In [ ]:
# get with lower thresholds
from neural_data.utils import get_all_saliency_map_ids_and_patches, get_saliency_map_ids_and_patches


sm_ids, patches = get_saliency_map_ids_and_patches((R,C), KERNEL_COORDINATE, INPUT_LAYER_NAME, pos_thresh, neg_thresh)
# sm_ids, patches = get_all_saliency_map_ids_and_patches(
#     KERNEL_COORDINATE, INPUT_LAYER_NAME, pos_thresh, neg_thresh
# )
pw = [p * kernel for p in patches]
lin_pw = np.array([p.reshape(-1) for p in pw])
lin_patches = np.array([p.reshape(-1) for p in patches])

# SAE

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

y = 1.0
alphas = [0.5, 1.0, 2.0, 5.0, 1000]

x = np.linspace(-4, 4, 500)

fig, ax = plt.subplots(figsize=(8, 5))

for alpha in alphas:
    sigma = y / (1 + alpha * x**2)
    ax.plot(x, sigma, label=f"alpha={alpha}")

ax.set_title("Lorentzian: y / (1 + alpha * x²)")
ax.set_xlabel("x")
ax.set_ylabel("sigma(x)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
import torch.optim as optim
 
 
class VanillaSparseAutoencoderV1(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)
        # normalize decoder columns to unit norm (same as sklearn)
        self._normalize_decoder()
 
    def _normalize_decoder(self):
        with torch.no_grad():
            norms = self.decoder.weight.norm(dim=0, keepdim=True).clamp(min=1.0)
            self.decoder.weight.div_(norms)
 
    def forward(self, x):
        codes = torch.relu(self.encoder(x))
        recon = self.decoder(codes)
        return recon, codes
 
 
def train_sae_v1(X, n_components, alpha=0.1, beta=0.1, lr=1e-3, epochs=2000, batch_size=256, weights_init_sigma=None):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: sparsity penalty weight
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = VanillaSparseAutoencoderV1(input_dim, n_components)
    # if weights_init_sigma is not None:
    #     torch.nn.init.normal(model.decoder.weight, 0, weights_init_sigma)
    optimizer = optim.Adam(model.parameters(), lr=lr)
 
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        X_t = X_t[idx]
 
        epoch_loss = 0
        for i in range(0, n_samples, batch_size):
            batch = X_t[i:i+batch_size]
            recon, codes = model(batch)
 
            recon_loss = ((batch - recon) ** 2).mean()
            sparsity_loss = alpha * codes.abs().mean()
            loss = recon_loss + sparsity_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            model._normalize_decoder()
 
            epoch_loss += loss.item()
 
        if epoch % 200 == 0:
            print(f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} sparse_loss {sparsity_loss:.4f}")
 
    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
def overlap_penalty_v2(W, alpha):
    tot_sum = 0
    for dim in range(W.shape[1]):
        for i in range(W.shape[0]):
            wi_sq = W[i, dim]**2
            prev_s = sum(W[j, dim]**2 for j in range(i))
            tot_sum += wi_sq + alpha * prev_s
    return tot_sum

def overlap_penalty_v2_vectorized(W, alpha):
    W_sq = W**2
    cumsum = torch.cumsum(W_sq, dim=0)
    prev_cumsum = cumsum - W_sq  # shift: sum of j < i
    return W_sq.sum() + alpha * prev_cumsum.sum()

In [ ]:
import torch.nn as nn
import torch.optim as optim
 

def overlap_penalty(W):
    tot_sum = 0
    for dim in range(W.shape[1]):
        w0_sq = W[0,dim]**2
        tot_sum += w0_sq
        for i in range(1, W.shape[0]):
            # 1 -> 72
            # prev weights sum
            prev_s = 0
            for j in range(i):
                prev_s += W[j, dim]**2
            tot_sum += (W[i,dim]**2) * prev_s
    return tot_sum
 
def tukka(W):
    W_sq = W ** 2
    W_abs = W.abs()

    cum_abs = W_abs.cumsum(dim=0)
    prev_abs = torch.roll(cum_abs, 1, 0)
    prev_abs[0] = 0

    tot_sum = (W_sq * prev_abs).sum() + W_sq[0].sum()
    return tot_sum

def overlap_penalty_vectorized(W):
    W_sq = W ** 2
    col_sum_sq = W_sq.sum(dim=0)
    col_sum_q4 = (W_sq ** 2).sum(dim=0)

    tot_sum = 0.5 * (col_sum_sq ** 2 - col_sum_q4).sum() + W_sq[0, :].sum()
    return tot_sum

def cosine_penalty(W):
    G = W.T @ W               # gram matrix, (11, 11)
    off_diag = G**2 - torch.diag(torch.diag(G**2))
    penalty = off_diag.sum()
    return penalty
 

class VanillaSparseAutoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)
 
 
    def forward(self, x):
        codes = torch.relu(self.encoder(x))
        recon = self.decoder(codes)
        return recon, codes
 
 
def train_sae(X, n_components, alpha=0.1, beta=0.1, lr=1e-3, epochs=2000, batch_size=256, weights_init_sigma=None):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: sparsity penalty weight
    """
    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = VanillaSparseAutoencoder(input_dim, n_components)
    if weights_init_sigma is not None:
        torch.nn.init.normal_(model.decoder.weight, 0, weights_init_sigma)
    optimizer = optim.Adam(model.parameters(), lr=lr)
 
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        X_t = X_t[idx]

        epoch_loss = 0
        for i in range(0, n_samples, batch_size):
            batch = X_t[i:i+batch_size]
            recon, codes = model(batch)
 
            recon_loss = beta*((batch - recon) ** 2).mean()
            overlap_loss = overlap_penalty_v2_vectorized(model.decoder.weight, 1000.)
            loss = recon_loss + overlap_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
 
            epoch_loss += loss.item()
 
        if epoch % 200 == 0:
            print(f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} overlap_loss {overlap_loss:.4f}")
 
    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
def phi_weight(W, c, k, alpha):
    s = 0
    for i in range(k+1):
        s += W[c][i]**2
    return torch.tensor(alpha*s + 1)

def overlap_penalty_v3(W, alpha, beta, gamma, delta):
    s = 0
    C, K = W.shape
    for c in range(C):
        for k in range(K):
            comp1 = beta * gamma * W[c][k]**2 * phi_weight(W, c, k-1, alpha)
            comp2 = beta * gamma * delta * torch.log(phi_weight(W, c, k-1, alpha))
            s += comp1 + comp2
    return s

def overlap_penalty_v3_vec(W, alpha, beta, gamma, delta):
    # W: (C, K)
    W_sq = W ** 2  # (C, K)
    
    # phi_weight(W, c, k-1, alpha) = alpha * sum(W[c, 0:k]**2) + 1
    # cumsum shifted right by 1 (k=0 gets sum of empty prefix = 0)
    cumsum = torch.cumsum(W_sq, dim=1)          # (C, K), cumsum[c,k] = sum W[c,0..k]^2
    phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
    phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1

    # comp1 = beta * gamma * W_sq * phi           # (C, K)
    # comp2 = beta * gamma * delta * torch.log(phi)  # (C, K)
    
    comp1 = W_sq * phi           # (C, K)
    comp2 = torch.log(phi)  # (C, K)
    return (comp1 - comp2).sum()

def all_penalties(batch, recon, W, codes, alpha, beta, gamma, delta):
    # recon_loss = gamma*delta*((batch - recon) ** 2).sum()
    recon_loss = ((batch - recon) ** 2).sum()
    overlap_loss = beta*overlap_penalty_v3_vec(W, alpha, beta, gamma, delta)
    # codes_loss = beta*delta*(codes**2).sum()
    codes_loss = beta*(codes**2).sum()
    return recon_loss, overlap_loss, codes_loss

In [ ]:
class GroupingAutoencoder(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

        self.sigma_x = nn.Parameter(torch.tensor(0.3).float())
        self.sigma_x.requires_grad_()
        self.sigma_s = nn.Parameter(torch.tensor(0.3).float())
        self.sigma_s.requires_grad_()
        self.mu_s = nn.Parameter(torch.tensor(0.).float())
        self.mu_s.requires_grad_()
        self.sigma_0 = nn.Parameter(torch.tensor(0.3).float())
        self.sigma_0.requires_grad_()
 
 
    def forward(self, x):
        codes = torch.relu(self.encoder(x))
        recon = self.decoder(codes)
        return recon, codes

    def loss_fn(self, x, recons, codes):
        # recons_loss = self._recons_loss(x, recons)
        recons_loss = self._gauss_loss(x, recons, self.sigma_x)
        codes_loss = self._gauss_loss(codes, self.mu_s, self.sigma_s)
        weights_loss = self._weights_loss()

        return recons_loss, codes_loss, weights_loss

    def _gauss_loss(self, x, mean, std):
        sq_sigma = std**2
        loss = ((x - mean) ** 2)
        loss /= sq_sigma
        loss += torch.log(2*torch.pi*sq_sigma)
        return loss.sum()

    def _weights_loss(self):
        W = self.decoder.weight
        # W_sq = W
        W_sq = W ** 2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)          # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1

        sigma_sq = self.sigma_0**2
        comp1 = (W_sq * phi) / sigma_sq
        comp2 = -torch.log(phi)
        comp3 = torch.log(2*torch.pi*sigma_sq)

        return (comp1 + comp2 + comp3).sum()

# def overlap_penalty_v3_vec(W, alpha, beta, gamma, delta):
#     # W: (C, K)
#     W_sq = W ** 2  # (C, K)
#     cumsum = torch.cumsum(W_sq, dim=1)          # (C, K), cumsum[c,k] = sum W[c,0..k]^2
#     phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
#     phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
    
#     # comp1 = beta * gamma * W_sq * phi           # (C, K)
#     # comp2 = beta * gamma * delta * torch.log(phi)  # (C, K)
    
#     comp1 = W_sq * phi           # (C, K)
    # comp2 = torch.log(phi)  # (C, K)
    # return (comp1 - comp2).sum()

In [ ]:
class GroupingAutoencoderFixedSigma(nn.Module):
    def __init__(self, input_dim, n_components):
        super().__init__()
        self.encoder = nn.Linear(input_dim, n_components)
        self.decoder = nn.Linear(n_components, input_dim, bias=False)

    def forward(self, x):
        codes = torch.relu(self.encoder(x))
        recon = self.decoder(codes)
        return recon, codes

    def loss_fn(self, x, recons, codes, alpha, beta, gamma, delta):
        recons_loss = self._gauss_loss(x, recons)

        codes_loss = beta * self._gauss_loss(codes, 0)

        weights_loss = delta*self._weights_loss(alpha, gamma)

        return recons_loss, codes_loss, weights_loss

    def _gauss_loss(self, x, mean):
        loss = ((x - mean) ** 2)
        return loss.sum()

    def _weights_loss(self, alpha, gamma):
        W = self.decoder.weight
        # W_sq = W
        W_sq = W ** 2  # (C, K)
        cumsum = torch.cumsum(W_sq, dim=1)          # (C, K), cumsum[c,k] = sum W[c,0..k]^2
        phi = alpha * torch.roll(cumsum, 1, dims=1) + 1  # (C, K)
        phi[:, 0] = 1  # k=0: phi_weight(W, c, -1, alpha) = alpha*0 + 1
        comp1 = (W_sq * phi)
        comp2 = -torch.log(phi)
        comp2 *= gamma
        return (comp1 + comp2).sum()

In [ ]:
def get_alpha(epoch, total_epochs, alpha_start=0.0, alpha_end=1.0):
    return alpha_start + (alpha_end - alpha_start) * (epoch / total_epochs)

def train_grouping_autoencoder_fixed_sigma(
    X, n_components, max_alpha=5000, beta=1., gamma=1., delta=1., lr=1e-3, epochs=2000, batch_size=256
):
    """
    X: numpy array (n_samples, input_dim)
    n_components: number of dictionary atoms
    alpha: lorentzian sigma shrinker parameter
    beta: codes regularizer coefficient
    gamma: log(phi) coefficient
    delta: atomic weights regularizer coefficient
    """
    # data needs to be centered and put to sigma_data
    # orig_mean, orig_std = X.mean(), X.std()
    # # scale to unit variance and 0 mean
    # X = ((X - orig_mean) / orig_std)
    # # scale to sigma_data
    # X *= sigma_data

    # see if this is useful, but oh well
    # sigma_x = (sigma_x / orig_std) * sigma_data

    X_t = torch.tensor(X, dtype=torch.float32)
    n_samples, input_dim = X_t.shape
 
    model = GroupingAutoencoderFixedSigma(input_dim, n_components)
    # torch.nn.init.normal_(model.decoder.weight, 0, 1)
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    model.decoder.weight.data = torch.tensor(Vt[:n_components].T, dtype=torch.float32)
    optimizer = optim.Adam(model.parameters(), lr=lr)
 
    # print(f"START TRAINING: alpha={alpha} sigma_x={sigma_x} sigma_s={sigma_s} sigma_data={sigma_data}")
    for epoch in range(epochs):
        # shuffle
        idx = torch.randperm(n_samples)
        X_t = X_t[idx]

        epoch_loss = 0
        for i in range(0, n_samples, batch_size):
            batch = X_t[i:i+batch_size]
            recon, codes = model(batch)
 

            alpha = get_alpha(epoch, epochs, 100, max_alpha)
            recon_loss, codes_loss, weight_loss = model.loss_fn(batch, recon, codes, alpha, beta, gamma, delta)
            loss = recon_loss + weight_loss + codes_loss
 
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                model.decoder.weight.data = torch.nn.functional.normalize(model.decoder.weight.data, dim=0)
 
            epoch_loss += loss.item()
 
        if epoch % 200 == 0:
            print(f"epoch {epoch:4d} | recon_loss {recon_loss:.4f} weight_loss {weight_loss:.4f} codes_loss {codes_loss:.4f}")
 
    # final codes and reconstruction
    with torch.no_grad():
        recon, codes = model(X_t)
 
    return (
        model,
        codes.numpy(),
        model.decoder.weight.T.detach().numpy(),  # (n_components, input_dim)
        recon.numpy(),
    )

In [ ]:
import numpy as np
import torch
import torch.optim as optim

def solve_codes(X, W, beta):
    """
    Closed form ridge regression for S update (fixed W).
    X: (n_samples, input_dim)
    W: (n_components, input_dim)
    beta: L2 regularization on codes
    Returns S: (n_samples, n_components)
    """
    WWT = W @ W.T  # (n_components, n_components)
    reg = WWT + beta * np.eye(W.shape[0])
    S = X @ W.T @ np.linalg.inv(reg)
    return S


def weights_loss(W, alpha, gamma, delta):
    """
    W: (n_components, input_dim) torch tensor with grad
    """
    W_sq = W ** 2
    cumsum = torch.cumsum(W_sq, dim=1)
    phi = alpha * torch.roll(cumsum, 1, dims=1) + 1
    phi[:, 0] = 1
    comp1 = W_sq * phi
    comp2 = -gamma * torch.log(phi)
    return delta * (comp1 + comp2).sum()


def recon_loss(X_batch, S_batch, W):
    """
    X_batch: (batch, input_dim) torch tensor
    S_batch: (batch, n_components) torch tensor
    W: (n_components, input_dim) torch tensor
    """
    recon = S_batch @ W
    return ((X_batch - recon) ** 2).sum()


def train_dict_learning(
    X, n_components,
    max_alpha=5000, beta=1., gamma=1., delta=1.,
    lr=1e-3, epochs=2000, batch_size=256
):
    n_samples, input_dim = X.shape

    # initialize W as a torch tensor with grad
    U, s, Vt = np.linalg.svd(X, full_matrices=False)
    W_init = Vt[:n_components]  # (n_components, input_dim)
    # W = torch.nn.init.normal_(
    #     torch.empty(n_components, input_dim), 0, 1
    # ).requires_grad_(True)
    W = torch.tensor(W_init).float()
    W.requires_grad_()

    optimizer = optim.Adam([W], lr=lr)

    def get_alpha(epoch):
        return 100 + (max_alpha - 100) * (epoch / epochs)

    for epoch in range(epochs):
        # --- S update: closed form (fixed W) ---
        W_np = W.detach().numpy()
        S = solve_codes(X, W_np, beta)

        # --- W update: gradient steps (fixed S) ---
        idx = torch.randperm(n_samples)
        X_t = torch.tensor(X[idx.numpy()], dtype=torch.float32)
        S_t = torch.tensor(S[idx.numpy()], dtype=torch.float32)

        alpha = get_alpha(epoch)

        for i in range(0, n_samples, batch_size):
            x_batch = X_t[i:i+batch_size]
            s_batch = S_t[i:i+batch_size]

            r_loss = recon_loss(x_batch, s_batch, W)
            w_loss = weights_loss(W, alpha, gamma, delta)
            loss = r_loss + w_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if epoch % 200 == 0:
            W_np = W.detach().numpy()
            recon = S @ W_np
            mse = np.mean((X - recon) ** 2)
            print(f"epoch {epoch:4d} | mse {mse:.4f} | w_loss {w_loss.item():.4f}")

    W_final = W.detach().numpy()
    S_final = solve_codes(X, W_final, beta)
    recon_final = S_final @ W_final

    return S_final, W_final, recon_final

In [ ]:
def make_dim_partition(patch_dim, n_components, seed=42):
    rng = np.random.RandomState(seed)
    perm = rng.permutation(patch_dim)
    return [list(perm[i::n_components]) for i in range(n_components)]

def generate_synthetic_patches(patch_dim=72, n_components=10, k=3, n_samples=1000, noise_std=0.01, seed=42):
    rng = np.random.RandomState(seed)
    
    dim_partition = make_dim_partition(patch_dim, n_components, seed)
    
    # ground truth atoms, nonzero only on owned dims
    W_true = np.zeros((n_components, patch_dim))
    for i, dims in enumerate(dim_partition):
        W_true[i, dims] = rng.randn(len(dims))
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)
    
    # each sample uses exactly k atoms
    codes_true = np.zeros((n_samples, n_components))
    for i in range(n_samples):
        idx = rng.choice(n_components, k, replace=False)
        codes_true[i, idx] = rng.randn(k)
    
    X = codes_true @ W_true
    X += rng.randn(*X.shape) * noise_std
    
    return X, W_true, codes_true, dim_partition

def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)
    
    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)
    
    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(axis=0)  # for each true atom, best cosine with any learned atom
    
    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered

In [ ]:
X, W_true, codes_true, dim_partition = generate_synthetic_patches(9, 3)
# X = (X - X.mean()) / X.std()
# X.shape

In [ ]:
# # DATA_SIGMA = X.std()
# # print(DATA_SIGMA)
# # DATA_SIGMA = X.std()

# # weirdly, the atoms are coming, but the reconstruction is shit
# # very very weird
# # insanely weird
# # hmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmmm

# orig_mean, orig_std = X.mean(), X.std()
# TARGET_STD = 0.4
# scaled_x = ((X - orig_mean) / orig_std) * TARGET_STD

torch.manual_seed(30)
model_v2, codes_v2, components_v2, recons_v2= train_grouping_autoencoder_fixed_sigma(
    X, 3, epochs=10_000, gamma=1., delta=1.3, beta=0.5, max_alpha=10_000
)

In [ ]:
S([recons[0].reshape(3,3), X[0].reshape(3,3)], mode=MODE)

In [ ]:
# w_scaled = ((W_true - X.mean()) / X.std()) * 0.3
evaluate_recovery(components, W_true)

In [ ]:
cs = [c for c in components_v2]
S([c.reshape(3,3) for c in np.concat([components_v2, w_scaled])], ncols=3, mode=MODE)

In [ ]:
show_72_list(W_true, mode=MODE)

In [ ]:
parts = make_dim_partition(72, 10)
for p in parts:
    print([a.item() for a in p])

In [ ]:
len(parts), sum([len(p) for p in parts])

In [8]:
import numpy as np

def generate_synthetic_patches(patch_dim=72, n_atoms=10, k=3, n_samples=1000, noise_std=0.01, seed=42):
    """
    patch_dim: length of each patch vector
    n_atoms: number of ground truth dictionary atoms
    k: number of atoms that combine per patch
    n_samples: number of patches to generate
    noise_std: gaussian noise added to each patch
    
    returns:
        X: (n_samples, patch_dim) generated patches
        W_true: (n_atoms, patch_dim) ground truth atoms (unit norm)
        codes_true: (n_samples, n_atoms) ground truth codes
    """
    rng = np.random.RandomState(seed)
    
    # ground truth atoms, unit normed
    W_true = rng.randn(n_atoms, patch_dim)
    W_true /= np.linalg.norm(W_true, axis=1, keepdims=True)
    
    # codes: each sample uses exactly k atoms
    codes_true = np.zeros((n_samples, n_atoms))
    for i in range(n_samples):
        idx = rng.choice(n_atoms, k, replace=False)
        codes_true[i, idx] = rng.randn(k)
    
    X = codes_true @ W_true
    X += rng.randn(*X.shape) * noise_std
    
    return X, W_true, codes_true


def evaluate_recovery(W_learned, W_true, threshold=0.95):
    """
    W_learned: (n_atoms, patch_dim)
    W_true: (n_atoms, patch_dim)
    
    for each true atom, finds the best matching learned atom by cosine similarity
    returns fraction of true atoms recovered above threshold
    """
    W_l = W_learned / (np.linalg.norm(W_learned, axis=1, keepdims=True) + 1e-8)
    W_t = W_true / (np.linalg.norm(W_true, axis=1, keepdims=True) + 1e-8)
    
    sim = np.abs(W_l @ W_t.T)  # (n_atoms, n_atoms), abs because sign is arbitrary
    best_match = sim.max(axis=0)  # for each true atom, best cosine with any learned atom
    
    recovered = (best_match >= threshold).mean()
    print(f"Mean best cosine similarity: {best_match.mean():.4f}")
    print(f"Fraction recovered (>{threshold}): {recovered:.4f}")
    return best_match, recovered


def _match_atoms(D1, D2):
    """
    Match atoms of D1 to atoms of D2 using the Hungarian algorithm
    on cosine distances. Returns (row_ind, col_ind, mean_cosine_similarity).
    
    D1, D2: shape (n_components, n_features) — sklearn's components_ layout.
    """
    # Cosine distance matrix: shape (n_components, n_components)
    cost = cosine_distances(D1, D2)          # values in [0, 2]
    row_ind, col_ind = linear_sum_assignment(cost)

    # Convert matched distances → similarities
    matched_similarities = 1 - cost[row_ind, col_ind]   # cosine similarity
    mean_sim = matched_similarities.mean()
    return row_ind, col_ind, matched_similarities, mean_sim


def hungarian_match(n_runs, all_components: list[torch.Tensor]):
    pairwise_sims = np.ones((n_runs, n_runs))   # diagonal = 1 by definition

    for i in range(n_runs):
        D_i = all_components[i]    # (n_components, n_features)
        for j in range(i + 1, n_runs):
            D_j = all_components[j]
            _, _, _, mean_sim = _match_atoms(D_i, D_j)
            pairwise_sims[i, j] = mean_sim
            pairwise_sims[j, i] = mean_sim      # symmetric

    # ── Global stability score ──────────────────────────────────────────────
    # Average over the upper triangle (all unique pairs)
    upper = pairwise_sims[np.triu_indices(n_runs, k=1)]
    stability_score = upper.mean()

    # ── Best run: highest average similarity to all other runs ─────────────
    mean_sim_per_run = pairwise_sims.mean(axis=1)   # includes self=1
    best_run_idx = int(np.argmax(mean_sim_per_run))

    return upper, stability_score, best_run_idx, pairwise_sims

In [ ]:
W = model_v2.decoder.weight  # (input_dim, n_components)
W_norm = W / (W.norm(dim=0, keepdim=True) + 1e-8)
gram = W_norm.T @ W_norm  # (n_components, n_components)
# diagonal is 1s, off-diagonal is cosine similarities

In [ ]:
plt.imshow(gram.detach().numpy(), cmap="gray")

In [ ]:
model, codes, components, recons = train_sae_v1(lin_pw, 3, alpha=0.01, epochs=2000)

In [ ]:
show_72_list([recons_v2[4], lin_pw[4]], mode=MODE)

In [ ]:
show_72_list([recons[0], lin_pw[0]], mode=MODE)

In [ ]:
n_samples, input_dim = lin_patches.shape
model = VanillaSparseAutoencoder(input_dim, 11)

In [ ]:
model.decoder.weight.shape

In [ ]:
W.shape

In [ ]:
model.decoder.weight.shape

In [ ]:
W = model.decoder.weight
tot_sum = 0
for dim in range(W.shape[1]):
    w0_sq = W[0,dim]**2
    tot_sum += w0_sq
    for i in range(1, model.decoder.weight.shape[0]):
        # 1 -> 72
        # prev weights sum
        prev_s = 0
        for j in range(i):
            prev_s += W[j, dim]**2
        tot_sum += (W[i,dim]**2) * prev_s

In [ ]:
# weight 0
tot_sum

In [ ]:
W_sq = W ** 2                          # (72, 11)
col_sum_sq = W_sq.sum(dim=0)           # (11,)
col_sum_q4 = (W_sq ** 2).sum(dim=0)   # (11,)

# sum over upper triangle pairs per dim, then sum over dims
tot_sum = 0.5 * (col_sum_sq ** 2 - col_sum_q4).sum()

In [ ]:
tot_sum

In [ ]:
W_sq = W ** 2
col_sum_sq = W_sq.sum(dim=0)
col_sum_q4 = (W_sq ** 2).sum(dim=0)

tot_sum = 0.5 * (col_sum_sq ** 2 - col_sum_q4).sum() + W_sq[0, :].sum()
tot_sum